# Milestone 4 — Logistic Regression and Random Forest baselines

Fit complete preprocessing/model pipelines on training data only. Compare validation precision, recall, F1, ROC-AUC, average precision (reported as PR-AUC), and confusion matrices at threshold 0.5. Average precision is the stepwise precision-recall summary, not trapezoidal area. No test performance is inspected.

In [1]:
from pathlib import Path
import nbformat
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data/raw/ai4i2020.csv").exists())
notebook = nbformat.read(PROJECT_ROOT / "notebooks/03_preprocessing.ipynb", as_version=4)
for cell in notebook.cells:
    if cell.cell_type == "code" and "definitions" in cell.metadata.get("tags", []):
        exec(compile(cell.source, "preprocessing_definitions", "exec"), globals())
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, fbeta_score, roc_auc_score, average_precision_score, confusion_matrix, brier_score_loss, precision_recall_curve, roc_curve
from sklearn.base import clone

In [2]:
def evaluate_probabilities(target, probabilities, threshold=0.5):
    predictions = np.asarray(probabilities) >= threshold
    matrix = confusion_matrix(target, predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    return {
        "threshold": float(threshold),
        "precision": float(precision_score(target, predictions, zero_division=0)),
        "recall": float(recall_score(target, predictions, zero_division=0)),
        "f1": float(f1_score(target, predictions, zero_division=0)),
        "f2": float(fbeta_score(target, predictions, beta=2, zero_division=0)),
        "roc_auc": float(roc_auc_score(target, probabilities)) if len(np.unique(target)) == 2 else None,
        "pr_auc": float(average_precision_score(target, probabilities)) if np.sum(target) else None,
        "brier_score": float(brier_score_loss(target, probabilities)),
        "confusion_matrix": matrix.tolist(),
        "false_negatives": int(fn),
        "false_negative_rate": float(fn / (fn + tp)) if fn + tp else None,
        "support": int(len(target)),
        "positive_support": int(np.sum(target)),
    }

def train_pipeline(estimator, features, target):
    pipeline = Pipeline([("preprocess", make_preprocessor()), ("classifier", estimator)])
    pipeline.fit(features, target)
    return pipeline


def select_threshold(target, probabilities, minimum_recall=0.90):
    precision, recall, thresholds = precision_recall_curve(target, probabilities)
    eligible = np.flatnonzero(recall[:-1] >= minimum_recall)
    best = max(eligible, key=lambda i: (precision[i], thresholds[i]))
    return float(thresholds[best])

In [3]:
records, manifest = load_partitions()
X_train, y_train = get_partition(records, manifest, "train")
X_validation, y_validation = get_partition(records, manifest, "validation")
candidates = {
    "logistic_regression": LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=1, n_jobs=4, random_state=RANDOM_STATE),
}
results = []
for name, estimator in candidates.items():
    pipeline = train_pipeline(estimator, X_train, y_train)
    probabilities = pipeline.predict_proba(X_validation)[:, 1]
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    result = {"model": name, "partition": "validation", **evaluate_probabilities(y_validation, probabilities)}
    results.append(result)
    joblib.dump(pipeline, MODEL_DIR / f"{name}.joblib")
(METRICS_DIR / "baselines.json").write_text(json.dumps(results, indent=2) + "\n")
display(pd.DataFrame(results).drop(columns="confusion_matrix"))
for result in results:
    print(result["model"], "confusion matrix:", result["confusion_matrix"])

,model,partition,threshold,precision,recall,f1,f2,roc_auc,pr_auc,brier_score,false_negatives,false_negative_rate,support,positive_support
0,logistic_regression,validation,0.5,0.727273,0.235294,0.355556,0.272109,0.873714,0.432080,0.024883,52,0.764706,2000,68
1,random_forest,validation,0.5,0.852941,0.426471,0.568627,0.473856,0.955795,0.745368,0.016188,39,0.573529,2000,68


logistic_regression confusion matrix: [[1926, 6], [52, 16]]
random_forest confusion matrix: [[1927, 5], [39, 29]]


## Evaluation checks

In [4]:
measured = evaluate_probabilities([0, 0, 1, 1], [0.1, 0.7, 0.4, 0.9])
assert measured["confusion_matrix"] == [[1, 1], [1, 1]]
assert measured["recall"] == 0.5 and measured["precision"] == 0.5
threshold = select_threshold(y_validation, probabilities)
assert evaluate_probabilities(y_validation, probabilities, threshold)["recall"] >= 0.90
restored = joblib.load(MODEL_DIR / "random_forest.joblib")
np.testing.assert_allclose(restored.predict_proba(X_validation), pipeline.predict_proba(X_validation))
print("Metric checks and saved pipeline round-trip passed.")

Metric checks and saved pipeline round-trip passed.
